# Geography SumLevels and Scopes

morpc-census describes any geographic query using two concepts:

- **Scope** — *where* to fetch data (e.g. the 15-county MORPC region, Franklin County, the state of Ohio)
- **SumLevel** — *at what resolution* (e.g. county, census tract, block group)

Both are represented as typed dataclasses so that function signatures are explicit and call sites have IDE support.

In [ ]:
from morpc_census import Scope, SumLevel, SCOPES, valid_scale

## Scope

A `Scope` wraps the Census API query parameters that define a geographic extent.  It has three fields:

| Field | Type | Description |
|---|---|---|
| `name` | `str` | Human-readable identifier (e.g. `"franklin"`) |
| `for_param` | `str` | Value for the Census API `for=` parameter |
| `in_param` | `str \| None` | Value for the `in=` parameter (optional) |

The `.params` property returns the ready-to-use dict for a Census API call.

In [ ]:
# Construct a Scope directly
franklin = Scope(name="franklin", for_param="county:049", in_param="state:39")
franklin

In [ ]:
# .params returns the dict passed to Census API calls
franklin.params

In [ ]:
# A scope with no in_param (e.g. the whole US)
us = Scope(name="us", for_param="us:1")
us.params

### The built-in SCOPES dictionary

`SCOPES` is a `dict[str, Scope]` pre-populated with:
- `"us"` — the entire United States
- `"columbuscbsa"` — the Columbus CBSA
- All 50 state names (e.g. `"Ohio"`, `"Indiana"`)
- All Ohio county names, lowercased (e.g. `"franklin"`, `"delaware"`)
- MORPC region names: `"region15"`, `"region10"`, `"region7"`, `"regioncbsa"`, `"regionmpo"`, etc.

In [ ]:
# All available scope names
list(SCOPES.keys())

In [ ]:
# Retrieve a built-in scope
SCOPES["franklin"]

In [ ]:
# The 15-county MORPC region — for_param is a comma-separated list of county FIPS codes
SCOPES["region15"]

In [ ]:
# .params is the dict you would pass directly to a Census API call
SCOPES["region15"].params

## SumLevel

A `SumLevel` pairs a Census API query name with its three-digit summary level code.

| Field | Type | Example |
|---|---|---|
| `name` | `str` | `"county"`, `"tract"`, `"block group"` |
| `sumlevel` | `str` | `"050"`, `"140"`, `"150"` |

`SumLevel` is frozen (immutable) and supports equality comparison.

In [ ]:
# Construct a SumLevel directly
county_scale = SumLevel(name="county", sumlevel="050")
county_scale

In [ ]:
tract_scale = SumLevel(name="tract", sumlevel="140")
tract_scale

### valid_scale()

`valid_scale(name)` looks up the scale name in `morpc.SUMLEVEL_DESCRIPTIONS` and returns the matching `SumLevel` object, or raises `ValueError` if the name is not recognized.  Use it when you have a scale name as a string and need to resolve the summary level code.

In [ ]:
# Resolve a scale name to a SumLevel object
valid_scale("tract")

In [ ]:
# Unrecognized names raise ValueError with the list of available options
try:
    valid_scale("neighborhood")
except ValueError as e:
    print(e)

## Fetching geometries with Scope and SumLevel

`fetch_geos_from_scale_scope(scope, scale)` accepts scope and scale *names* (strings) and returns a GeoDataFrame.  Internally it resolves the `Scope` and `SumLevel` objects and constructs the Census TIGERweb query.

> **Note:** The cells below make live network calls to the Census API and TIGERweb REST API.

In [ ]:
from morpc_census import fetch_geos_from_scale_scope

# Fetch county boundaries for the 15-county MORPC region
geos = fetch_geos_from_scale_scope(scope="region15")
geos

In [ ]:
geos.plot()

In [ ]:
# Fetch census tracts within Franklin County
tracts = fetch_geos_from_scale_scope(scope="franklin", scale="tract")
tracts.plot()